In [292]:
from google.cloud import bigquery

client = bigquery.Client(project="wagon-bootcamp-501612-i1")

query = """
SELECT * EXCEPT(ingredients)
FROM `wagon-bootcamp-501612-i1.recipes_clean_300.recipes_final_array`
"""

df = client.query(query).to_dataframe()

In [300]:

test = df.iloc[456]


In [294]:
df.head()

,name,ingredients_raw,steps,servings,persons,portion_size,ingredients_clean,type_dish,type_diet,type_meal,type_occasion,type_origin,time_to_make
0,Half the Sodium Seasoned Salt,"['1 1/2 tablespoons salt', '1 teaspoon ...",['Combine all ingredients in a small bowl and ...,45.0,1,0 g,"[powdered sugar, paprika, curry powder, onion ...",[],"[dietary, low-protein, low-cholesterol, health...",[],[],[],15
1,Provoleta (Argentina),"['1 inch thick slice provolone cheese', ' ...",['Heat a cast-iron skillet over very high heat...,4.0,1,0 g,"[cheese, oregano, bell pepper]",[],[],[appetizers],[],"[south-american, argentine]",15
2,Cake,"['1 teaspoon vanilla extract', '1 teasp...",['Mix together all cake ingredients and place ...,10.0,1,0 g,"[vanilla, nutmeg, cinnamon]","[cakes, vegetables]","[dietary, low-protein, low-sodium, gluten-free...","[desserts, lunch]",[],[],15
3,Russian Tomato Salad,"[' tomatoes (homegrown or other tasty)', '1 -...",['Slice the tomatoes into thick (approx. 3/8”)...,1.0,1,0 g,"[tomato, garlic, mayonnaise, dill]","[salads, vegetables]","[dietary, low-protein, low-carb, low-sodium, v...","[brunch, side-dishes]","[brunch, summer]","[european, russian]",15
4,Scrambled Egg Beaters and Ham,"['1/4 cup Egg Beaters egg substitute', '1/...","['In a microwavable bowl, place egg beaters an...",1.0,1,0 g,"[egg, cheese, pork]",[],[],[breakfast],[],[],15


In [286]:
user =  {'time_max': '15',
 'occasion': ['summer'],
 'dish': [],
 'meal': ['main-dish'],
 'diet': ['gluten-free', 'dietary'],
 'origin': [],
 'pantry_items': ['chocolate', 'butter', 'chocolate', 'lemon juice']}

In [287]:
user2 =  {'time_max': "0",
 'occasion': [],
 'dish': [],
 'meal': [],
 'diet': [],
 'origin': [],
 'pantry_items': []}

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from numpy import hstack
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import ParameterGrid


def predict_closer(user , df_clean):
    #On créé les mlb pour chaque colonnes à encoder (permettra de faire des 'groupes de colonnes avec le même poids, tout en choisissant les poids')
    mlb_dish = MultiLabelBinarizer()
    mlb_diet = MultiLabelBinarizer()
    mlb_meal = MultiLabelBinarizer()
    mlb_occasion = MultiLabelBinarizer()
    mlb_origin = MultiLabelBinarizer()
    mlb_time_to_make = MultiLabelBinarizer()


    param_grid = {'dish':[3], 'diet':[5], 'meal':[1], 'occasion':[1],
              'origin':[1], 'time_to_make':[3]}

    grid = list(ParameterGrid(param_grid))


    # On applique les mlb aux colonnes
    X_dish = mlb_dish.fit_transform(df_clean['type_dish'])
    X_diet = mlb_diet.fit_transform(df_clean['type_diet'])
    X_meal = mlb_meal.fit_transform(df_clean['type_meal'])
    X_occasion = mlb_occasion.fit_transform(df_clean['type_occasion'])
    X_origin = mlb_origin.fit_transform(df_clean['type_origin'])
    X_time_to_make = mlb_time_to_make.fit_transform(df_clean['time_to_make'])

    history = get_history()

    if not user["dish"] and not user["origin"] and user["time_max"] == "0" and not user["diet"] and not user["meal"] and not user["occasion"]:
        X_test_dish = mlb_dish.transform([[max(set(history["dish"]), key=history["dish"].count)]])
        X_test_diet = mlb_diet.transform([[max(set(history["diet"]), key=history["diet"].count)]])
        X_test_meal = mlb_meal.transform([[max(set(history["meal"]), key=history["meal"].count)]])
        X_test_occasion = mlb_occasion.transform([[max(set(history["occasion"]), key=history["occasion"].count)]])
        X_test_origin = mlb_origin.transform([[max(set(history["origin"]), key=history["origin"].count)]])
        X_test_time_to_make = mlb_time_to_make.transform([[max(set(history["time_max"]), key=history["time_max"].count)]])
    else:
        X_test_dish = mlb_dish.transform([user["dish"]])
        X_test_diet = mlb_diet.transform([user['diet']])
        X_test_meal = mlb_meal.transform([user["meal"]])
        X_test_occasion = mlb_occasion.transform([user['occasion']])
        X_test_origin = mlb_origin.transform([user['origin']])
        X_test_time_to_make = mlb_time_to_make.transform([user['time_max']])


    result = []
    for param in grid:
        X = hstack([X_dish * param['dish'],X_diet * param['diet'],X_meal*param['meal'],
                X_occasion * param['occasion'],X_origin * param['origin'],
                X_time_to_make * param['time_to_make']])


        X_test = hstack([X_test_dish * param['dish'], X_test_diet * param['diet'],X_test_meal*param['meal'],
                        X_test_occasion * param['occasion'],X_test_origin * param['origin'],
                        X_test_time_to_make * param['time_to_make']])

        model = NearestNeighbors(
            n_neighbors=5,
            metric="cosine")

        model.fit(X)
        distances, indices = model.kneighbors(X_test)
        result.append({
        **param,
        "score": distances,
        "recipe_index": indices
    })
    response = []
    for i in list(result[0]["recipe_index"][0]):
        dict = {}
        dict["name"] = str(df.iloc[i]["name"])
        dict["ingredients_raw"] = df.iloc[i]["ingredients_raw"]
        dict["steps"] = df.iloc[i]["steps"]
        dict["servings"] = int(df.iloc[i]["servings"])
        dict["persons"] = int(df.iloc[i]["persons"])
        dict["portion_size"] = df.iloc[i]["portion_size"]
        dict["ingredients_clean"] = list(df.iloc[i]["ingredients_clean"])
        dict["type_dish"] = list(df.iloc[i]["type_dish"])
        dict["type_diet"] = list(df.iloc[i]["type_diet"])
        dict["type_meal"] = list(df.iloc[i]["type_meal"])
        dict["type_occasion"] = list(df.iloc[i]["type_occasion"])
        dict["type_origin"] = list(df.iloc[i]["type_origin"])
        dict["time_to_make"] = df.iloc[i]["time_to_make"]
        response.append(dict)
    return response


In [317]:
result = predict_closer(user , df)
result

[{'name': 'Simply Avocado',
  'ingredients_raw': "['1       avocado, ripe ', '1   tablespoon    extra virgin olive oil', '1/2      lime', '1   teaspoon    sea salt']",
  'steps': "['slice avocado', 'drizzle olive oil over', 'squeeze juice of 1/2 a lime.', 'sprinkle with sea salt.', 'garnish with cilantro (optional).', 'serve.']",
  'servings': 2,
  'persons': 1,
  'portion_size': '127 g',
  'ingredients_clean': ['avocado', 'cooking oil', 'salt'],
  'type_dish': [],
  'type_diet': ['dietary', 'gluten-free'],
  'type_meal': ['main-dish'],
  'type_occasion': [],
  'type_origin': [],
  'time_to_make': '15'},
 {'name': 'Fast Thin Gluten Free Pizza',
  'ingredients_raw': "['2   tablespoons    almond flour', '2   dashes    italian seasoning', '1   dash    sea salt', '1   dash    onion powder', '1/4  cup    mozzarella cheese, grated ', '1   slice    tomatoes']",
  'steps': "['Move shelf in oven near the top then place oven on broil.', 'Grease a cookie sheet.', 'Sprinkle some almond flour onto 

In [301]:
for i in list(result[0]["recipe_index"][0]):
    print(df.iloc[i])
    print("_______________________")

name                                                    Simply Avocado
ingredients_raw      ['1       avocado, ripe ', '1   tablespoon    ...
steps                ['slice avocado', 'drizzle olive oil over', 's...
servings                                                           2.0
persons                                                              1
portion_size                                                     127 g
ingredients_clean                         [avocado, cooking oil, salt]
type_dish                                                           []
type_diet                                       [dietary, gluten-free]
type_meal                                                  [main-dish]
type_occasion                                                       []
type_origin                                                         []
time_to_make                                                        15
Name: 7692, dtype: object
_______________________
name                       

In [315]:
response = []
for i in list(result[0]["recipe_index"][0]):
    dict = {}
    dict["name"] = str(df.iloc[i]["name"])
    dict["ingredients_raw"] = df.iloc[i]["ingredients_raw"]
    dict["steps"] = df.iloc[i]["steps"]
    dict["servings"] = int(df.iloc[i]["servings"])
    dict["persons"] = int(df.iloc[i]["persons"])
    dict["portion_size"] = df.iloc[i]["portion_size"]
    dict["ingredients_clean"] = list(df.iloc[i]["ingredients_clean"])
    dict["type_dish"] = list(df.iloc[i]["type_dish"])
    dict["type_diet"] = list(df.iloc[i]["type_diet"])
    dict["type_meal"] = list(df.iloc[i]["type_meal"])
    dict["type_occasion"] = list(df.iloc[i]["type_occasion"])
    dict["type_origin"] = list(df.iloc[i]["type_origin"])
    dict["time_to_make"] = df.iloc[i]["time_to_make"]
    response.append(dict)
response

[{'name': 'Simply Avocado',
  'ingredients_raw': "['1       avocado, ripe ', '1   tablespoon    extra virgin olive oil', '1/2      lime', '1   teaspoon    sea salt']",
  'steps': "['slice avocado', 'drizzle olive oil over', 'squeeze juice of 1/2 a lime.', 'sprinkle with sea salt.', 'garnish with cilantro (optional).', 'serve.']",
  'servings': 2,
  'persons': 1,
  'portion_size': '127 g',
  'ingredients_clean': ['avocado', 'cooking oil', 'salt'],
  'type_dish': [],
  'type_diet': ['dietary', 'gluten-free'],
  'type_meal': ['main-dish'],
  'type_occasion': [],
  'type_origin': [],
  'time_to_make': '15'},
 {'name': 'Fast Thin Gluten Free Pizza',
  'ingredients_raw': "['2   tablespoons    almond flour', '2   dashes    italian seasoning', '1   dash    sea salt', '1   dash    onion powder', '1/4  cup    mozzarella cheese, grated ', '1   slice    tomatoes']",
  'steps': "['Move shelf in oven near the top then place oven on broil.', 'Grease a cookie sheet.', 'Sprinkle some almond flour onto 

In [251]:
def get_history():
    history = {'time_max': ['15',"30", "15", "15"],
    'occasion': ['summer', "spring", "brunch", "spring"],
    'dish': ["vegetables","pasta" , "vegetables" , "vegetables"],
    'meal': ['main-dish', 'main-dish', 'main-dish', 'main-dish', 'dessert'],
    'diet': ['gluten-free', 'dietary', 'dietary', 'dietary' , 'dietary', 'gluten-free'],
    'origin': ['american', "european", "european", "european" , "american"]}
    return history